In [ ]:
# ================================================================
# NOTEBOOK : nb_silver_product
# Read from  : bronze_lakehouse → bronze_product
# Write to   : silver_lakehouse → silver_product
# ================================================================

StatementMeta(, 18a9e2a1-2fcd-4aa2-ad06-1aad60adf078, 11, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import trim, initcap,col,to_date,when

bronze_df = spark.sql("SELECT * FROM bronze_lakehouse.bronze_product")
print(f"[BRONZE] Rows:{bronze_df.count()}")

display(bronze_df)



StatementMeta(, 18a9e2a1-2fcd-4aa2-ad06-1aad60adf078, 12, Finished, Available, Finished, False)

[BRONZE] Rows:200


SynapseWidget(Synapse.DataFrame, ae61702c-399a-4cbf-815f-389e7b3da890)

In [ ]:
bronze_df.printSchema()


StatementMeta(, 18a9e2a1-2fcd-4aa2-ad06-1aad60adf078, 14, Finished, Available, Finished, False)

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- CostPrice: double (nullable = true)
 |-- ListPrice: double (nullable = true)
 |-- IsActive: boolean (nullable = true)
 |-- LaunchDate: date (nullable = true)



In [ ]:


silver_df = (bronze_df
# ── Nulls + Dedup ──────────────────────────────────────────
.filter(col("ProductID").isNotNull())
.filter(col("ListPrice").isNotNull())
.dropDuplicates(["ProductID"])

# ── Fix data types ─────────────────────────────────────────
.withColumn("CostPrice", col("CostPrice").cast("decimal(12,2)"))
.withColumn("ListPrice", col("ListPrice").cast("decimal(12,2)"))
.withColumn("LaunchDate", to_date(col("LaunchDate"), "yyyy-MM-dd"))
.withColumn("isActive",col("isActive").cast("boolean"))
 
# ── Standardise strings ────────────────────────────────────
.withColumn("Category",  initcap(trim(col("CostPrice"))))
.withColumn("SubCategory",  initcap(trim(col("SubCategory"))))
.withColumn("Brand"     ,initcap(trim(col("Brand"))))

# ── Derived columns ────────────────────────────────────────
# Gross margin percentage
.withColumn("Gross_marginpct",F.round((F.col("ListPrice") - F.col("CostPrice"))/F.col("ListPrice") * 100,2))

# Price band — useful for segmentation in reports
.withColumn("PriceBand",when(col("ListPrice") < 1000, "Budget")
.when(col("ListPrice") < 5000, "Mid-range")
.when(col("ListPrice") < 20000, "Premium")
.otherwise("Luxury")
)


# Days since launch (product maturity)
.withColumn("DaysSinceLunch", F.datediff(F.current_date(), col("LaunchDate")))


# New product flag (launched in last 180 days)
.withColumn("IsNewProduct", F.datediff(F.current_date(), col("LaunchDate")) <= 180)


# ── Metadata ───────────────────────────────────────────────
.withColumn("silver_load_ts", F.current_timestamp())

) 

silver_df.write.format("delta").mode("overwrite")\
.option("overwriteSchema", "true").saveAsTable("silver_product")


print(f"[DONE] Silver_product_written:{silver_df.count()} rows")

display(silver_df)




StatementMeta(, 18a9e2a1-2fcd-4aa2-ad06-1aad60adf078, 29, Finished, Available, Finished, False)

[DONE] Silver_product_written:200 rows


SynapseWidget(Synapse.DataFrame, d680ba33-7a1f-4b66-be93-168ac1fadeb7)

StatementMeta(, 18a9e2a1-2fcd-4aa2-ad06-1aad60adf078, 28, Finished, Available, Finished, False)

'silver_df'